# Validación Estadística — Tesis Jaime

Este cuaderno descarga automáticamente la sábana de datos, genera las gráficas de validación (`curva-aforo.png` y `error-diario-mae.png`) y ejecuta la prueba t-Student de la hipótesis general.

**Instrucciones:** presiona `Entorno de ejecución > Ejecutar todas` (o Ctrl+F9) para correr todo el cuaderno de principio a fin.

In [ ]:
# 0. Instalación e importación de librerías
!pip install -q pandas matplotlib scipy openpyxl gdown

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import requests

## 1. Carga de la sábana de datos

In [ ]:
# OPCIÓN A: Descargar desde GitHub (recomendado)
# Reemplaza con el link 'raw' de tu repositorio
URL_DATOS = "https://raw.githubusercontent.com/rojasjaimeis/Verificaci-n_Aforo/main/sabana-datos-v2.xlsx"

r = requests.get(URL_DATOS)
with open("sabana-datos-v2.xlsx", "wb") as f:
    f.write(r.content)

# OPCIÓN B: Descargar desde Google Drive (alternativa)
# import gdown
# FILE_ID = "TU_FILE_ID_DE_DRIVE"
# gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", "sabana-datos-v2.xlsx", quiet=False)

df = pd.read_excel("sabana-datos-v2.xlsx")
print(f"Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

## 2. Vista general de los datos

In [ ]:
print(df.columns.tolist())
df.describe()

## 3. Curva de Aforo

**Ajusta** `COL_NIVEL` y `COL_CAUDAL` con los nombres reales de tus columnas.

In [ ]:
COL_NIVEL = "Nivel_m"        # <-- reemplaza con el nombre real de tu columna
COL_CAUDAL = "Caudal_m3s"    # <-- reemplaza con el nombre real de tu columna

plt.figure(figsize=(9, 6))
plt.scatter(df[COL_NIVEL], df[COL_CAUDAL], alpha=0.6, edgecolor="k", s=40)
plt.xlabel("Nivel (m)")
plt.ylabel("Caudal (m³/s)")
plt.title("Curva de Aforo")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("curva-aforo.png", dpi=300)
plt.show()

## 4. Error Diario (MAE)

**Ajusta** `COL_FECHA`, `COL_OBSERVADO` y `COL_PREDICHO` con los nombres reales.

In [ ]:
COL_FECHA = "Fecha"                 # <-- ajusta
COL_OBSERVADO = "Caudal_Observado"  # <-- ajusta
COL_PREDICHO = "Caudal_Modelo"      # <-- ajusta

df[COL_FECHA] = pd.to_datetime(df[COL_FECHA])
df["Error_Absoluto"] = (df[COL_OBSERVADO] - df[COL_PREDICHO]).abs()

mae_diario = df.groupby(df[COL_FECHA].dt.date)["Error_Absoluto"].mean()

plt.figure(figsize=(11, 5))
plt.plot(mae_diario.index, mae_diario.values, marker="o", linewidth=1)
plt.xlabel("Fecha")
plt.ylabel("MAE (m³/s)")
plt.title("Error Absoluto Medio (MAE) Diario")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("error-diario-mae.png", dpi=300)
plt.show()

print(f"MAE promedio del periodo: {df['Error_Absoluto'].mean():.4f}")

## 5. Prueba t-Student de la Hipótesis General

Se usa una **prueba t pareada** (`ttest_rel`) porque compara dos mediciones relacionadas —observado vs. modelo— sobre las mismas fechas.

- **H0:** no existe diferencia significativa entre los valores observados y los predichos por el modelo.
- **H1:** existe una diferencia significativa entre ambos.

Si tu hipótesis compara dos grupos independientes en vez de mediciones pareadas, reemplaza `ttest_rel` por `stats.ttest_ind(grupo1, grupo2)`.

In [ ]:
t_stat, p_value = stats.ttest_rel(df[COL_OBSERVADO], df[COL_PREDICHO])

print(f"Estadístico t: {t_stat:.4f}")
print(f"Valor p: {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print("Se rechaza H0: existe diferencia estadísticamente significativa (p < 0.05)")
else:
    print("No se rechaza H0: no hay diferencia estadísticamente significativa (p >= 0.05)")

## 6. Conclusión

_Completa aquí la interpretación de los resultados en función del valor p obtenido y del contexto de tu hipótesis general._